# Qwen2.5-3B LoRA fine-tune -- predictive keyboard

Mirrors `scripts/train_transformer.py` + `scripts/infer_transformer.py` from the repo,
inlined so this notebook is self-contained on Kaggle (no repo/network access to those
files). **Continuation run**: resumes the previous run's LoRA adapter (loss was
plateauing around step ~1000-1160), trains 6h more on the same data chunk, then
inference on a stratified 10% dev sample (not the full set) for a quick check.

**Inputs expected as Kaggle notebook add-ons:**
- Dataset `predictive_keyboard`: `train_final.src.tok` + `dev_set_final.csv`
- Model `qwen2.5` (transformers/3b variant) -- local path, no HF download/token needed
- Dataset `qwen3b-lora-ckpt`: previous run's LoRA adapter -- resumed from if attached,
  else this falls back to a fresh adapter automatically

**Outputs (this notebook's /kaggle/working/, becomes the notebook's Output on commit):**
- `qwen3b_lora/` -- the trained LoRA adapter + tokenizer
- `loss_curve.png` -- training loss vs step
- `dev_predictions_finetuned.csv` + printed accuracy table on the stratified 10% sample

In [ ]:
# Unsloth's own requirement: import it before trl/transformers/peft, or its
# monkeypatches apply incorrectly (hit this bug locally -- corrupts SFTConfig.eos_token
# via a to_dict() redaction round-trip). One cell, one shot, order preserved.
!pip install -q -U unsloth


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Unsloth's free/OSS build refuses multi-GPU
                                            # outright (raises NotImplementedError if it
                                            # sees >1 GPU) -- this Kaggle session has a
                                            # T4x2 accelerator, so pin to GPU0 before any
                                            # torch/unsloth import. Unsloth can't use the
                                            # second T4 anyway; a single T4 is what we'd
                                            # have gotten with a plain "GPU T4" shape.

from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from transformers import (
    TrainerCallback, AutoModelForCausalLM, AutoTokenizer,
    LogitsProcessor, StoppingCriteria,
)


In [ ]:
import glob, os, json

print("/kaggle/input tree:")
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + "/")

def find_file(name):
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert matches, f"{name} not found under /kaggle/input -- check the dataset is attached"
    return matches[0]

def find_qwen_base():
    # Don't guess the mount path -- model_sources attachment naming isn't guaranteed to
    # match the "/kaggle/input/qwen2.5/transformers/3b/1" shape a UI attach shows. Find
    # it by content instead: any config.json whose model_type is qwen2.
    for cfg_path in glob.glob("/kaggle/input/**/config.json", recursive=True):
        try:
            cfg = json.load(open(cfg_path))
        except (json.JSONDecodeError, OSError):
            continue
        if cfg.get("model_type") == "qwen2":
            return os.path.dirname(cfg_path)
    return None

def find_lora_adapter():
    # Same content-based find, for a previous run's LoRA checkpoint (attached as its own
    # dataset) instead of guessing a mount path. None if not attached -- fresh run then.
    matches = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    return os.path.dirname(matches[0]) if matches else None

TRAIN_PATH = find_file("train_final.src.tok")
DEV_PATH = find_file("dev_set_final.csv")
MODEL_PATH = find_qwen_base()
assert MODEL_PATH, "no config.json with model_type=='qwen2' found under /kaggle/input -- check the model is attached (see tree above)"
RESUME_PATH = find_lora_adapter()

OUT_DIR = "/kaggle/working/qwen3b_lora"
print("TRAIN_PATH:", TRAIN_PATH)
print("DEV_PATH:", DEV_PATH)
print("MODEL_PATH:", MODEL_PATH)
print("RESUME_PATH:", RESUME_PATH or "(none -- fresh adapter)")


## Train -- 6h wall-clock cap, plain causal LM, packed sequences

Same objective as `train_transformer.py`: every position in every packed sequence is a supervised target (no `[letter]` tag -- the first-letter constraint lives entirely in the inference logit mask below). `--no-4bit` equivalent (`load_in_4bit=False`): T4 has 16GB, no need to pay the QLoRA quant/dequant overhead for a 3B model.

In [ ]:
MAX_LINES = 900_000   # buffer sized for ~6h at ~1200 tok/s (measured locally on a 4060,
                      # QLoRA/4bit) -- T4 bf16/fp16 LoRA should be faster, not slower, so this
                      # is a floor, not a tight fit. If training finishes the file before 6h,
                      # the trainer just starts a second epoch over the same lines (HF Trainer's
                      # own behavior when max_steps > one epoch's worth) -- TimeLimit below still
                      # caps wall time regardless.
MAX_HOURS = 6.0        # continuation run, bumped from the 2h checkpoint run -- loss was
                       # plateauing around step ~1000-1160 there, but that run was still inside
                       # warmup_steps=30's ramp for most of its length (see warmup_steps comment
                       # below) so the plateau reading is unreliable; 6h buys enough steps past
                       # full warmup to see the real trend
SKIP_LINES = 900_000   # moving past the first chunk -- last run only reached 53% of one epoch
                       # over it (1160 steps), but packing shuffles all 900_000 lines into
                       # sequences before iterating, so a partial epoch is ~random coverage of
                       # the whole chunk, not a clean prefix -- no way to skip just the unseen
                       # 47%. Skipping the entire chunk to genuinely fresh lines instead.
TRAIN_SEED = 44        # bumped again for this run. SFTConfig defaults to seed=42, and a
                       # fresh Trainer/session resets global_step to 0 -- with the same seed over
                       # the same packed dataset in the same order, epoch-0's "random" shuffle is
                       # the IDENTICAL permutation every time, so a from-scratch continuation run
                       # would silently replay the exact sequences (in the exact order) the last
                       # run already trained on instead of reaching new ones. Different seed here
                       # = different shuffle = actually-new coverage of this chunk.

def load_lines(path, max_lines, skip=0):
    with open(path, encoding="utf-8") as f:
        for _ in range(skip):
            next(f, None)
        if max_lines:
            lines = [next(f, "").strip() for _ in range(max_lines)]
        else:
            lines = [line.strip() for line in f]
    return [line for line in lines if line]

lines = load_lines(TRAIN_PATH, MAX_LINES, SKIP_LINES)
print(f"{len(lines)} lines loaded")
ds = Dataset.from_dict({"text": lines})


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    RESUME_PATH or MODEL_PATH, max_seq_length=512, load_in_4bit=False,
)
if not RESUME_PATH:
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
else:
    print("resumed LoRA adapter from", RESUME_PATH)

# T4 (Turing) has no fast bf16 tensor cores (needs Ampere+) -- detect instead of assuming.
bf16_ok = torch.cuda.is_bf16_supported()
print("bf16 supported:", bf16_ok)


In [ ]:
import time, json

class TimeLimit(TrainerCallback):
    """Wall-clock stop -- packed-sequence count per session is hard to predict exactly,
    wall time isn't. Fires on_step_end so the run always saves a valid checkpoint after."""
    def __init__(self, max_hours):
        self.deadline = time.time() + max_hours * 3600 if max_hours else None

    def on_step_end(self, args, state, control, **kwargs):
        if self.deadline and time.time() >= self.deadline:
            control.should_training_stop = True
        return control

class JsonlLogger(TrainerCallback):
    """Durable, granular progress: trainer_state.json only gets written every
    save_steps (500) -- this appends+flushes every logging_steps (20) instead, so
    /kaggle/working/train_log.jsonl is current even if the run is killed mid-checkpoint.
    Also prints so it shows up live in the Kaggle log viewer."""
    def __init__(self, path):
        self.f = open(path, "a")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return
        row = {"step": state.global_step, "elapsed_s": round(time.time() - t0, 1), **logs}
        print(row, flush=True)
        self.f.write(json.dumps(row) + "\n")
        self.f.flush()

t0 = time.time()
trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=SFTConfig(
        output_dir=OUT_DIR,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        max_steps=100_000,        # safety cap; TimeLimit is the real stopper
        warmup_steps=30,          # absolute count, not Unsloth's default warmup_steps=0.1 (a
                                  # FRACTION of max_steps on transformers>=5.0) -- confirmed from
                                  # run 1's logged lr: 0.1 * max_steps(100_000) = 10,000-step
                                  # warmup, and the whole ~1160-step run never got past 11% of
                                  # that ramp (peak lr ~5e-5 backed out from lr=5.8e-6 at step
                                  # 1160), which is why the loss curve looked flat throughout --
                                  # LR was still near-zero the entire time, never mind cooldown
        max_length=512,
        packing=True,
        dataset_text_field="text",
        bf16=bf16_ok,
        fp16=not bf16_ok,
        logging_steps=20,
        save_steps=500,           # periodic checkpoint under OUT_DIR -- survives a mid-run kill
        save_total_limit=2,
        seed=TRAIN_SEED,          # see TRAIN_SEED comment above -- must differ per continuation run
        report_to=[],
    ),
    callbacks=[TimeLimit(MAX_HOURS), JsonlLogger("/kaggle/working/train_log.jsonl")],
)
trainer.train()
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("saved LoRA adapter ->", OUT_DIR)


In [ ]:
import matplotlib.pyplot as plt

log = [e for e in trainer.state.log_history if "loss" in e]
steps = [e["step"] for e in log]
losses = [e["loss"] for e in log]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses)
plt.xlabel("step")
plt.ylabel("training loss")
plt.title(f"Qwen2.5-3B LoRA fine-tune ({len(steps)} logged steps)")
plt.tight_layout()
plt.savefig("/kaggle/working/loss_curve.png", dpi=120)
plt.show()


## Inference -- full dev set, KV-cache `generate()`

Mirrors `infer_transformer.py`: first generated piece is logit-masked to the given first letter (boundary-marker-aware, works for either SentencePiece `▁` or GPT2-BPE `Ġ`), then `model.generate()` continues with its native KV cache (no manual recompute loop -- this is the ~4.7x-faster rewrite from the repo's history) until the next word-boundary piece or EOS. Symbol rows are trivial (EDA ceiling ~99.6%). Number rows go through the model too (`--mask-number` scheme from `report.md`'s Qwen2.5-vs-Qwen3.5 check): every number answer here is anonymized to `"1"*length`, so the prediction is masked to `"1"*len` before scoring -- a length-only check, no n-gram, no ensemble. Word/number/symbol print as separate rows below, easy to read independently.

In [ ]:
FastLanguageModel.for_inference(model)  # Unsloth's fast native-KV-cache inference path
model.generation_config.max_length = None  # silence the harmless "both max_new_tokens and
                                            # max_length are set" warning -- max_new_tokens
                                            # already wins, this just stops the log spam


In [ ]:
import csv, re

CAT_NUMBER = re.compile(r"[0-9]+")
CAT_WORD = re.compile(r"(?=.*[a-z])[a-z']+", re.IGNORECASE)  # >=1 real letter -- a bare "'" isn't a word

def categorize(tok):
    if CAT_NUMBER.fullmatch(tok):
        return "number"
    if CAT_WORD.fullmatch(tok):
        return "word"
    return "symbol"

def detect_boundary(tokenizer):
    vocab = tokenizer.get_vocab()
    counts = {"▁": 0, "Ġ": 0}
    for piece in vocab:
        if piece[:1] in counts:
            counts[piece[:1]] += 1
    return max(counts, key=counts.get)

def build_letter_masks(tokenizer, boundary, device, vocab_size):
    vocab = tokenizer.get_vocab()
    letters = list("abcdefghijklmnopqrstuvwxyz0123456789")
    masks = {c: torch.zeros(vocab_size, dtype=torch.bool) for c in letters}
    for piece, idx in vocab.items():
        if len(piece) > 1 and piece[0] == boundary and piece[1].lower() in masks:
            masks[piece[1].lower()][idx] = True
    return {c: m.to(device) for c, m in masks.items()}

def get_boundary_ids(tokenizer, boundary):
    return [idx for piece, idx in tokenizer.get_vocab().items() if piece.startswith(boundary)]

class FirstTokenLetterMask(LogitsProcessor):
    def __init__(self, letter_mask, prompt_len):
        self.letter_mask = letter_mask
        self.prompt_len = prompt_len

    def __call__(self, input_ids, scores):
        if input_ids.shape[1] == self.prompt_len:
            scores = scores.masked_fill(~self.letter_mask, float("-inf"))
        return scores

class StopAtWordBoundary(StoppingCriteria):
    def __init__(self, prompt_len, boundary_ids_tensor, eos_id):
        self.prompt_len = prompt_len
        self.boundary_ids_tensor = boundary_ids_tensor
        self.eos_id = eos_id

    def __call__(self, input_ids, scores, **kwargs):
        cur_len = input_ids.shape[1]
        if cur_len <= self.prompt_len + 1:
            return torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)
        last = input_ids[:, -1]
        return torch.isin(last, self.boundary_ids_tensor) | (last == self.eos_id)

@torch.inference_mode()
def predict_batch(model, tokenizer, contexts, letters, masks, boundary_ids_tensor, device, max_extra=4):
    enc = tokenizer(contexts, return_tensors="pt", padding=True).to(device)
    letter_mask = torch.stack([masks[l.lower()] for l in letters])
    prompt_len = enc["input_ids"].shape[1]
    eos_id = tokenizer.eos_token_id

    out = model.generate(
        **enc,
        max_new_tokens=max_extra + 1,
        do_sample=False,
        logits_processor=[FirstTokenLetterMask(letter_mask, prompt_len)],
        stopping_criteria=[StopAtWordBoundary(prompt_len, boundary_ids_tensor, eos_id)],
        pad_token_id=eos_id,
    )
    generated = out[:, prompt_len:].tolist()

    boundary_set = set(boundary_ids_tensor.tolist())
    words = []
    for row in generated:
        cut = next((i for i, tid in enumerate(row) if i > 0 and (tid in boundary_set or tid == eos_id)), len(row))
        words.append(tokenizer.decode(row[:cut]).strip())
    return words


In [ ]:
import random

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
device = "cuda" if torch.cuda.is_available() else "cpu"

boundary = detect_boundary(tokenizer)
vocab_size = model.get_output_embeddings().weight.shape[0]
masks = build_letter_masks(tokenizer, boundary, device, vocab_size)
boundary_ids_tensor = torch.tensor(get_boundary_ids(tokenizer, boundary), device=device)

with open(DEV_PATH, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(f"{len(rows)} dev rows total")

SAMPLE_FRAC = 0.10  # quick continuation-check inference, not the full dev set -- stratified
                    # by category so word/symbol/number stay proportionally represented
                    # instead of a plain random sample maybe skewing the small number bucket
random.seed(42)
by_cat = {}
for r in rows:
    by_cat.setdefault(categorize(r["answer"]), []).append(r)
source_sizes = {cat: len(group) for cat, group in by_cat.items()}
rows = []
for cat, group in by_cat.items():
    k = max(1, round(len(group) * SAMPLE_FRAC))
    rows.extend(random.sample(group, k))
print(f"stratified {SAMPLE_FRAC*100:.0f}% sample: {len(rows)} rows (source sizes: {source_sizes})")

alpha_rows = [r for r in rows if r["first letter"].isalpha()]
digit_rows = [r for r in rows if r["first letter"].isdigit()]
symbol_rows = [r for r in rows if not r["first letter"].isalnum()]

results = []
for r in symbol_rows:
    results.append((r, r["first letter"]))

# --mask-number scheme (matches the Qwen2.5 vs Qwen3.5 report.md comparison): number
# rows go through the model exactly like alpha rows, not a shortcut guess. Scoring
# masks the prediction to "1"*len below -- length-only check, digits don't matter,
# since every number answer in this dataset IS "1"*length (anonymized).
alpha_rows = alpha_rows + digit_rows


In [ ]:
BATCH_SIZE = 32  # T4 16GB has more headroom than the local 4060's 8GB

t0 = time.time()
for start in range(0, len(alpha_rows), BATCH_SIZE):
    batch = alpha_rows[start:start + BATCH_SIZE]
    contexts = [r["context"] for r in batch]
    letters = [r["first letter"] for r in batch]
    preds = predict_batch(model, tokenizer, contexts, letters, masks, boundary_ids_tensor, device)
    for r, p in zip(batch, preds):
        results.append((r, p))
    done = start + len(batch)
    elapsed = time.time() - t0
    if done % (BATCH_SIZE * 20) == 0 or done == len(alpha_rows):
        print(f"[{done}/{len(alpha_rows)} alpha rows] {elapsed:.1f}s elapsed, {done/max(elapsed,1e-9):.2f} rows/s", flush=True)


In [ ]:
cat_correct, cat_total = {}, {}
with open("/kaggle/working/dev_predictions_finetuned.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["context", "first_letter", "answer", "category", "prediction", "correct"])
    for r, pred in results:
        cat = categorize(r["answer"])
        scored_pred = pred.strip()
        if cat == "number":
            scored_pred = "1" * len(scored_pred)
        correct = scored_pred.lower() == r["answer"].strip().lower()
        cat_total[cat] = cat_total.get(cat, 0) + 1
        cat_correct[cat] = cat_correct.get(cat, 0) + correct
        w.writerow([r["context"], r["first letter"], r["answer"], cat, pred, correct])

print(f"\n--- fine-tuned Qwen2.5-3B accuracy, stratified {SAMPLE_FRAC*100:.0f}% dev sample ---")
tot_c = tot_n = 0
for cat in ("word", "symbol", "number"):
    c, n = cat_correct.get(cat, 0), cat_total.get(cat, 0)
    tot_c += c
    tot_n += n
    if n:
        print(f"{cat:8s} {c:6d} / {n:6d}  {c/n*100:.2f}%")
print(f"{'overall':8s} {tot_c:6d} / {tot_n:6d}  {tot_c/tot_n*100:.2f}%")
